# Track reconstruction — fit a muon from its Cherenkov light

Given the per-PMT charge and time of a real (PhotonSim) event, recover the muon's
**energy, vertex, direction and start time** by gradient-based optimization of the
differentiable forward model.

This notebook is a thin wrapper over the reconstruction kernel in `lucid.fitting` /
`lucid.optimization` — the **same** staged pipeline (energy scan → grid + time
multilateration seeds → cone direction → Fisher–Gauss–Newton two-start fit) that the
`lucid-optimize` CLI and the S3DF batch jobs call, so single-event and many-event
reconstruction never drift apart.

| stage | seam |
|---|---|
| read a PhotonSim event | `sources.event_io.read_photon_data_from_photonsim` + `pad_photon_data` |
| energy / vertex / direction seeds | `optimization` grid + cone search, `fitting.seed_vertex_time` |
| the fit | `fitting.fit_track_multistart` (+ `ReconModel`) |

> **Prerequisite:** the example ROOT file comes from `./scripts/download_data.sh`.
> The two `fit_track_multistart` cells run several minutes each with no per-iteration output — that's normal.


## 1. Build the data + prediction simulators

Two views of the same engine: a **data** simulator (realistic readout, reads ROOT photons)
produces the observed event; a **prediction** simulator (differentiable, per-photon) is
what the fit takes gradients of.

In [ ]:
import sys; sys.path.append('..')
import numpy as np, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
from lucid.geometry import generate_detector
from lucid.simulation import setup_event_simulator
from lucid.detector_params import load_detector_params
from lucid.sources.event_io import read_photon_data_from_photonsim, pad_photon_data
from lucid.fitting import (ReconModel, fit_track_multistart, track_from_vec9, vec9_from_track,
                           vec9_dir, seed_vertex_time)
from lucid.fitting.sweep import DEFAULT_RECIPE as RECIPE   # the exact validated Fisher-GN recipe
from lucid.optimization.grid_search import hierarchical_position_grid_search, get_detector_bounds
from lucid.optimization.utils.functions import (hierarchical_direction_search_cone,
                                                energy_scan_optimization)

GEOM = '../config/SK_like_geom_config.json'
PHYS = '../config/SK_like_physics_config.json'
ROOT = '../data/water/muon/1000MeV_100events.root'
K, NBUF, NPH, TTS = 8, 400_000, 250_000, 2.5
EV = 7   # which PhotonSim event to reconstruct

det = generate_detector(GEOM); ND = len(det.all_points); POS = np.asarray(det.all_points)
bounds = get_detector_bounds(det)
dp = load_detector_params(PHYS, num_sensors=ND)
dp = dp._replace(response=dp.response._replace(tts=jnp.asarray(TTS)))   # 2.5 ns transit-time spread

# Emitter config = the validated recipe (matches `lucid.fitting.sweep`): sample model wavelengths
# over the net's GEANT4 emission band [1.84, 4.51] eV, and use importance seed sampling so the full
# wide-angle emission tail is kept (the default seed_mode is already 'importance'; set explicitly so
# this notebook is robust to the code default).
CHER_BAND = (274.91, 673.83)

data_sim = setup_event_simulator(GEOM, NBUF, temperature=None, K=K, is_data=True, hit_mode='realistic',
                                 physics_config=PHYS, default_detector_params=dp, particle='muon',
                                 wavelength_mode=True, apply_smearing=False)
pred = setup_event_simulator(GEOM, NPH, temperature=0.1, K=K, hit_mode='per_photon', physics_config=PHYS,
                             default_detector_params=True, particle='muon', wavelength_mode=True,
                             pos_grad_threshold=K, n_grad_iters=K, cherenkov_emission_band=CHER_BAND)
model = ReconModel(pred, ND, sigma=TTS, delta=1.0, time_weight=RECIPE['time_weight'],
                  energy_from_scale=True, energy_scale_mode='simtotal')  # energy from total charge
REC_FIT = {k: v for k, v in RECIPE.items() if k != 'time_weight'}   # fit_track knobs
print(f'{ND} PMTs')

## 2. Make an observed event (with a known truth)

We read a PhotonSim muon, apply a fixed rotation+shift so the truth vertex/direction are
known but non-trivial, and run the data simulator to get the observed per-PMT charge & time.

In [ ]:
def rotax(u, deg):
    a = np.radians(deg); ca, sa = np.cos(a), np.sin(a); u = u / np.linalg.norm(u)
    ux = np.array([[0,-u[2],u[1]],[u[2],0,-u[0]],[-u[1],u[0],0]])
    return np.eye(3)*ca + sa*ux + (1-ca)*np.outer(u, u)

R = rotax(np.array([0.3, 0.9, 0.]), 50.); sh = np.array([4.0, -3.0, 6.0]) * 100.   # cm
raw = read_photon_data_from_photonsim(ROOT, EV)
O = np.asarray(raw['photon_origins']).astype(float); c = O.mean(0)
raw = dict(raw); raw['photon_origins'] = (O - c) @ R.T + c + sh
raw['photon_directions'] = np.asarray(raw['photon_directions']).astype(float) @ R.T
vtx_true = ((np.zeros(3) - c) @ R.T + c + sh) / 100.0          # exact gun vertex, m
dir_true = np.array([0., 0., 1.]) @ R.T                        # exact gun direction

pd, _ = pad_photon_data(raw, NBUF)
dummy = track_from_vec9(jnp.array([1000.,0,0,0, 0.,1., 0.,1., 0.]))   # is_data ignores the track
c_, t_ = jax.lax.stop_gradient(data_sim(dummy, jax.random.PRNGKey(7000+EV), pd))
oc = np.asarray(c_); ot = np.where(oc > 0, np.asarray(t_), 0.)
print(f'event {EV}: {int((oc>0).sum())} PMTs lit, {oc.sum():.0f} total charge')
print(f'truth: vtx={np.round(vtx_true,2)} m  dir={np.round(dir_true,3)}')

## 3. Two complementary seeds

Charge constrains the vertex *across* the track but is degenerate *along* it; first-arrival
time is the opposite. So we seed from both a **charge grid search** and a **time
multilateration**, and let the fit keep whichever wins.

In [ ]:
ocf, otf, POSf = jnp.asarray(oc), jnp.asarray(ot), jnp.asarray(POS)
e0 = energy_scan_optimization(pred, jnp.zeros(3), jnp.arccos(1/jnp.sqrt(3)), jnp.pi/4, 0.,
                              POSf, otf, ocf, (ocf, otf), 1000., 700., 12, 0)['best_energy']

def make_seed(vtx, t0g):
    c2 = hierarchical_direction_search_cone(pred, jnp.asarray(vtx), t0g, POSf, otf, ocf,
                                            (ocf, otf), e0, 3, 8, 90., 0.5, 0)
    dg = np.array([np.sin(c2['best_theta'])*np.cos(c2['best_phi']),
                   np.sin(c2['best_theta'])*np.sin(c2['best_phi']), np.cos(c2['best_theta'])])
    return vec9_from_track(e0, np.asarray(vtx), dg, t0=t0g)

p1 = hierarchical_position_grid_search(POSf, otf, ocf, jnp.zeros(3), 0., 0., bounds,
                                       n_div=5, t0_n_div=5, levels=6, verbosity=0)
seedA = make_seed(np.asarray(p1['best_position']), float(p1['best_t0']))   # charge-grid
seedB = make_seed(*seed_vertex_time(POS, oc, ot))                          # time-multilateration
print(f'seed A (charge) vtx err {np.linalg.norm(seedA[1:4]-vtx_true)*100:5.0f} cm')
print(f'seed B (time)   vtx err {np.linalg.norm(seedB[1:4]-vtx_true)*100:5.0f} cm')

## 4. Fit and report

`fit_track_multistart` runs Fisher–Gauss–Newton from both seeds and returns the better one.

In [ ]:
truth9 = np.asarray(vec9_from_track(1000., vtx_true, dir_true, t0=0.))
# fit_track_multistart(verbose=True) prints the seed-selection table + the track result table
res, MS = fit_track_multistart(model, oc, ot, [seedA, seedB], verbose=True, truth=truth9,
                               **REC_FIT)
d_fit = vec9_dir(res)

In [ ]:
hist = MS['per_seed'][MS['which']][1]; tr = hist['traj']; it = np.arange(len(tr))
verr = np.linalg.norm(tr[:, 1:4] - vtx_true, axis=1) * 100
dirs = np.stack([vec9_dir(v) for v in tr]); aerr = np.degrees(np.arccos(np.clip(dirs @ dir_true, -1, 1)))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(it, verr); ax[0].set(xlabel='GN iter', ylabel='vertex err (cm)', title='vertex'); ax[0].grid(alpha=.3)
ax[1].plot(it, aerr, color='C1'); ax[1].set(xlabel='GN iter', ylabel='dir err (deg)', title='direction'); ax[1].grid(alpha=.3)
ax[2].semilogy(it, hist['gnorm'], color='C2'); ax[2].set(xlabel='GN iter', ylabel='‖g‖', title='gradient norm'); ax[2].grid(alpha=.3)
ax[2].axvline(hist['best_iter'], ls='--', c='k', lw=.7, label='min‖g‖ iter (diagnostic)'); ax[2].legend()
vfit = np.linalg.norm(res[1:4] - vtx_true) * 100
afit = float(np.degrees(np.arccos(np.clip(d_fit @ dir_true, -1, 1))))
fig.suptitle(f'two-start recon, event {EV} — fit vtx {vfit:.1f} cm / dir {afit:.2f}° (readout = Polyak avg)')
fig.tight_layout(); plt.show()

## Takeaways

- One muon is reconstructed to **~cm vertex** and **~degree direction** by differentiating
  the forward model — no template fits.
- The **two-start** seeding (charge + time) covers the complementary degeneracies; the fit
  keeps the better basin.
- This exact `fit_track_multistart` call is the shared kernel — point it at many events on
  S3DF for a full resolution study without changing the logic.

## Reconstruct on a sphere (JUNO)

The same seed → Fisher-GN fit works **unchanged** on a non-cylinder detector — only the config
files and `detector_type` change. We reconstruct the *same* fabricated muon in a JUNO-like
spherical detector (17.5 m radius, water). The seeder (`get_detector_bounds` /
`hierarchical_position_grid_search`) is geometry-aware, so nothing else moves. (Lighter photon
count here — this is a generality demo, not a precision run.)

In [ ]:
GEOM_S, PHYS_S = '../config/JUNO_geom_config.json', '../config/JUNO_physics_config.json'
NPH_S = 150_000
det_s = generate_detector(GEOM_S); ND_s = len(det_s.all_points); POS_s = jnp.asarray(det_s.all_points)
bounds_s = get_detector_bounds(det_s)
dp_s = load_detector_params(PHYS_S, num_sensors=ND_s)
dp_s = dp_s._replace(response=dp_s.response._replace(tts=jnp.asarray(TTS)))
data_s = setup_event_simulator(GEOM_S, NBUF, temperature=None, K=K, is_data=True, hit_mode='realistic',
             detector_type='sphere', physics_config=PHYS_S, default_detector_params=dp_s,
             particle='muon', wavelength_mode=True, apply_smearing=False)
pred_s = setup_event_simulator(GEOM_S, NPH_S, temperature=0.1, K=K, hit_mode='per_photon',
             detector_type='sphere', physics_config=PHYS_S, default_detector_params=True,
             particle='muon', wavelength_mode=True, pos_grad_threshold=K, n_grad_iters=K,
             cherenkov_emission_band=CHER_BAND)
model_s = ReconModel(pred_s, ND_s, sigma=TTS, delta=1.0, time_weight=RECIPE['time_weight'],
                    energy_from_scale=True, energy_scale_mode='simtotal')

# reuse the SAME fabricated-truth event (rotated/shifted muon) from above
pd_s, _ = pad_photon_data(raw, NBUF)
c_s, t_s = jax.lax.stop_gradient(data_s(dummy, jax.random.PRNGKey(7007), pd_s))
oc_s = np.asarray(c_s); ot_s = np.where(oc_s > 0, np.asarray(t_s), 0.)
ocf_s, otf_s = jnp.asarray(oc_s), jnp.asarray(ot_s)

e0_s = energy_scan_optimization(pred_s, jnp.zeros(3), jnp.arccos(1/jnp.sqrt(3)), jnp.pi/4, 0.,
                                POS_s, otf_s, ocf_s, (ocf_s, otf_s), 1000., 700., 12, 0)['best_energy']
def seed_s(vtx, t0g):
    c2 = hierarchical_direction_search_cone(pred_s, jnp.asarray(vtx), t0g, POS_s, otf_s, ocf_s,
                                            (ocf_s, otf_s), e0_s, 3, 8, 90., 0.5, 0)
    dg = np.array([np.sin(c2['best_theta'])*np.cos(c2['best_phi']),
                   np.sin(c2['best_theta'])*np.sin(c2['best_phi']), np.cos(c2['best_theta'])])
    return vec9_from_track(e0_s, np.asarray(vtx), dg, t0=t0g)
p1s = hierarchical_position_grid_search(POS_s, otf_s, ocf_s, jnp.zeros(3), 0., 0., bounds_s,
                                        n_div=5, t0_n_div=5, levels=5, verbosity=0)
seedA_s = seed_s(np.asarray(p1s['best_position']), float(p1s['best_t0']))
seedB_s = seed_s(*seed_vertex_time(POS_s, oc_s, ot_s))
res_s, _ = fit_track_multistart(model_s, oc_s, ot_s, [seedA_s, seedB_s], **REC_FIT)
print(f'JUNO sphere (r={bounds_s["r"]} m): {int((oc_s>0).sum())} PMTs lit')
print(f'  vertex err {np.linalg.norm(res_s[1:4]-vtx_true)*100:6.2f} cm '
      f'| dir err {np.degrees(np.arccos(np.clip(vec9_dir(res_s)@dir_true,-1,1))):5.2f} deg '
      f'| E bias {res_s[0]-1000.:+6.1f} MeV')
print('Same pipeline, different geometry -- only GEOM/PHYS + detector_type changed.')